In [ ]:
# 04-latest (2024-12-01) - Added profit calculation (target_profit_bb) for SP-5
# No special packages needed - notebook 03 now saves predictions directly
# We just load the pre-computed predictions instead of loading sklearn models
print("SP-4 v2024-12-01: Loading pre-computed predictions from SP-3, computing profit targets")

In [ ]:
# SP-4: Advanced Feature Engineering Pipeline (PySpark Version)
# Uses SP-2 output and pre-computed SP-3 predictions (no model loading needed)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructType, StructField
import time
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

print("=" * 80)
print("SP-4: Advanced Feature Engineering Pipeline (PySpark)")
print("=" * 80)
start_time = time.time()

In [ ]:
# Configuration
WORKSPACE_DIR = '/Workspace/Users/leo.lwakabamba@gmail.com/poker-ml-data/'
UC_VOLUME_DIR = '/Volumes/pokerml/default/data/'

# Model Registry (Unity Catalog three-level namespace)
MODEL_REGISTRY_PREFIX = "pokerml.default"  # catalog.schema

# Input: SP-2 output (from UC Volume)
SP2_INPUT_PATH = UC_VOLUME_DIR + 'processed/sp2_player_events_labeled'

# SP-3 SparkML models directory (from UC Volume)
MODELS_DIR = UC_VOLUME_DIR + 'models/'

# Output path (to UC Volume)
OUTPUT_PATH = UC_VOLUME_DIR + 'processed/sp4_features_complete'

# Rolling window sizes for historical features
HIST_WINDOWS = [3, 5, 10]

# ============================================================================
# DEBUG MODE - Read from pipeline config (UC Volume - persists across Python restarts)
# ============================================================================
import json

# UC Volume path (persists across restarts, unlike /tmp)
CONFIG_PATH = '/Volumes/pokerml/default/data/pipeline_config.json'

try:
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    DEBUG_MODE = config.get('debug_mode', True)
    DEBUG_MAX_ROWS = config.get('max_rows', 10000)
    DEBUG_SAMPLE_FRACTION = config.get('sample_fraction', 0.01)
    print(f"   Loaded config from {CONFIG_PATH}")
    print(f"   DEBUG_MODE={DEBUG_MODE}, MAX_ROWS={DEBUG_MAX_ROWS:,}")
except FileNotFoundError:
    DEBUG_MODE = True
    DEBUG_SAMPLE_FRACTION = 0.01
    DEBUG_MAX_ROWS = 10000
    print(f"   Config not found at {CONFIG_PATH}, using defaults")
# ============================================================================

print(f"SP-2 Input: {SP2_INPUT_PATH}")
print(f"Models Directory: {MODELS_DIR}")
print(f"Model Registry: {MODEL_REGISTRY_PREFIX}")
print(f"Output: {OUTPUT_PATH}")
if DEBUG_MODE:
    print(f"\n*** DEBUG MODE ENABLED ***")
    print(f"    Sample fraction: {DEBUG_SAMPLE_FRACTION}")
    print(f"    Max rows: {DEBUG_MAX_ROWS:,}")
else:
    print(f"\n*** FULL MODE - processing all data ***")

In [ ]:
# STEP 1: Load SP-2 output and validate required columns
print("\n[1/8] Loading SP-2 output...")

player_events = spark.read.parquet(SP2_INPUT_PATH)
initial_count = player_events.count()
print(f"   Loaded {initial_count:,} player events from SP-2")

# ============================================================================
# SELECT TRACE_HAND_ID FOR PIPELINE VALIDATION
# ============================================================================
TRACE_HAND_ID = player_events.select('hand_id').first()['hand_id']
print(f"\n*** TRACE_HAND_ID selected: {TRACE_HAND_ID} ***")

# ============================================================================
# TRACE: RAW INPUT FROM SP-2
# ============================================================================
print(f"\n[TRACE] RAW INPUT from SP-2 for {TRACE_HAND_ID}:")
player_events.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'action_type',
    'position_from_button', 'position_name', 'num_players',
    'starting_stack', 'pot_size'
).show(20, truncate=False)

# Apply DEBUG sampling if enabled
if DEBUG_MODE:
    print(f"\n   Applying DEBUG sampling ({DEBUG_SAMPLE_FRACTION*100:.0f}%)...")
    sampled_count = int(initial_count * DEBUG_SAMPLE_FRACTION)
    target_count = min(sampled_count, DEBUG_MAX_ROWS)
    
    # Sample by hand_id to keep data consistent
    all_hands = player_events.select('hand_id').distinct()
    hands_count = all_hands.count()
    sample_frac = min(DEBUG_SAMPLE_FRACTION, DEBUG_MAX_ROWS / initial_count)
    sampled_hands = all_hands.sample(fraction=sample_frac, seed=42)
    
    # IMPORTANT: Make sure TRACE_HAND_ID is in the sample
    trace_hand_df = spark.createDataFrame([(TRACE_HAND_ID,)], ['hand_id'])
    sampled_hands = sampled_hands.union(trace_hand_df).distinct()
    
    player_events = player_events.join(sampled_hands, on='hand_id', how='inner')
    initial_count = player_events.count()
    print(f"   Sampled to {initial_count:,} rows (including TRACE_HAND_ID)")

print(f"   Columns: {len(player_events.columns)}")

# Show all columns for debugging
print("\n   All columns in dataset:")
for i, col in enumerate(sorted(player_events.columns)):
    print(f"      {col}")

# Define required columns
REQUIRED_COLUMNS = ['hand_id', 'actor', 'street', 'action_type']
POSITION_COLUMNS = ['seat_no']  # For position features
BOARD_COLUMNS = ['flop', 'turn', 'river']  # For board texture
STACK_COLUMNS = ['starting_stack', 'amount']  # For stack features

# Check required columns
missing_required = [c for c in REQUIRED_COLUMNS if c not in player_events.columns]
if missing_required:
    raise ValueError(f"FATAL: Missing required columns: {missing_required}")

# Check position columns
missing_position = [c for c in POSITION_COLUMNS if c not in player_events.columns]
if missing_position:
    print(f"\n   NOTE: Missing position columns: {missing_position}")
    print("   Will use position_from_button if available from SP-2")

# Check board columns  
missing_board = [c for c in BOARD_COLUMNS if c not in player_events.columns]
if missing_board:
    print(f"\n   NOTE: Missing raw board columns: {missing_board}")
    print("   Will use derived board features if available from SP-2")

# Check stack columns
missing_stack = [c for c in STACK_COLUMNS if c not in player_events.columns]
if missing_stack:
    print(f"\n   WARNING: Missing stack columns: {missing_stack}")

# ============================================================================
# TRACE: VERIFY TRACE_HAND_ID AFTER SAMPLING
# ============================================================================
print(f"\n[TRACE - AFTER SAMPLING] Verifying {TRACE_HAND_ID}:")
trace_count = player_events.filter(F.col('hand_id') == TRACE_HAND_ID).count()
print(f"   Rows for TRACE_HAND_ID: {trace_count}")

# Show sample data
print("\n   Sample data:")
player_events.select('hand_id', 'actor', 'street', 'action_type').show(5, truncate=False)

In [ ]:
# STEP 2: Load pre-computed opponent predictions from SP-3
print("\n[2/8] Loading pre-computed opponent predictions from SP-3...")

SP3_PREDICTIONS_PATH = UC_VOLUME_DIR + 'processed/sp3_opponent_predictions'

try:
    sp3_predictions = spark.read.parquet(SP3_PREDICTIONS_PATH)
    pred_count = sp3_predictions.count()
    print(f"   Loaded {pred_count:,} predictions from SP-3")
    
    # Show distribution
    print("\n   Prediction distribution:")
    sp3_predictions.groupBy('predicted_bucket').count().orderBy('count', ascending=False).show()
    
    # Show sample
    print("\n   Sample predictions:")
    sp3_predictions.show(5, truncate=False)
    
    # TRACE: Show predictions for TRACE_HAND_ID
    print(f"\n[TRACE] SP-3 predictions for {TRACE_HAND_ID}:")
    sp3_predictions.filter(F.col('hand_id') == TRACE_HAND_ID).show(20, truncate=False)
    
    SP3_PREDICTIONS_LOADED = True
    
except Exception as e:
    print(f"\n   ERROR: Could not load SP-3 predictions: {e}")
    print(f"   Expected path: {SP3_PREDICTIONS_PATH}")
    print("\n   To fix: Run notebook 03_OpponentModeling first, which now saves predictions directly.")
    print("   Then re-run this notebook.")
    SP3_PREDICTIONS_LOADED = False
    sp3_predictions = None

In [ ]:
# STEP 3: Position features
print("\n[3/8] Processing position features...")

# ============================================================================
# VERIFICATION: Track row count at start of step
# ============================================================================
rows_at_step_start = player_events.count()
print(f"   Rows at START of step 3: {rows_at_step_start:,}")

# Check if position features already exist from SP-2
has_position_from_button = 'position_from_button' in player_events.columns
has_position_name = 'position_name' in player_events.columns
has_num_players = 'num_players' in player_events.columns

if has_position_from_button and has_position_name:
    print("   Position features found in SP-2 output (from notebook 01/02)")
    
    # Check coverage
    total = player_events.count()
    with_position = player_events.filter(F.col('position_from_button').isNotNull()).count()
    print(f"   Position data coverage: {with_position:,}/{total:,} ({100*with_position/total:.1f}%)")
    
    # Use position_name directly as position_bucket
    player_events = player_events.withColumn('position_bucket', F.col('position_name'))
    
    # If position_from_button is null, fill with default
    player_events = player_events.fillna({'position_from_button': 3, 'position_name': 'MP'})
    
else:
    print("   WARNING: Position features not found in SP-2 output")
    print("   Re-run notebooks 01 and 02 to generate position data")
    print("   Using action order as approximation (less accurate)")
    
    # Fall back to action-order approximation
    action_order_window = Window.partitionBy('hand_id').orderBy(F.monotonically_increasing_id())
    player_events = player_events.withColumn('action_order', F.row_number().over(action_order_window))
    
    first_action = player_events.groupBy('hand_id', 'actor').agg(
        F.min('action_order').alias('first_action_order')
    )
    
    rows_before = player_events.count()
    player_events = player_events.join(first_action, on=['hand_id', 'actor'], how='left')
    rows_after = player_events.count()
    print(f"   After first_action join: {rows_before:,} -> {rows_after:,}")
    
    position_window = Window.partitionBy('hand_id').orderBy('first_action_order')
    player_events = player_events.withColumn(
        'position_index',
        F.dense_rank().over(position_window)
    )
    
    # Create approximate position bucket
    player_events = player_events.withColumn(
        'position_bucket',
        F.when(F.col('position_index') == 1, 'early')
        .when(F.col('position_index') == 2, 'middle')
        .otherwise('late')
    )
    
    # Add placeholder for position_from_button
    player_events = player_events.withColumn('position_from_button', F.lit(3))  # Default to middle
    
    player_events = player_events.drop('action_order', 'first_action_order')

# Compute players at table per hand if not already present
if 'players_at_table' not in player_events.columns:
    if has_num_players:
        # Use num_players from SP-2
        player_events = player_events.withColumn('players_at_table', F.col('num_players'))
    else:
        # Compute from data
        players_per_hand = player_events.groupBy('hand_id').agg(
            F.countDistinct('actor').alias('players_at_table')
        )
        rows_before = player_events.count()
        player_events = player_events.join(players_per_hand, on='hand_id', how='left')
        rows_after = player_events.count()
        print(f"   After players_per_hand join: {rows_before:,} -> {rows_after:,}")

print("   Position features ready")

# ============================================================================
# VERIFICATION: Check row count at end of step
# ============================================================================
rows_at_step_end = player_events.count()
print(f"   Rows at END of step 3: {rows_at_step_end:,}")
if rows_at_step_end != rows_at_step_start:
    print(f"   *** WARNING: Lost {rows_at_step_start - rows_at_step_end:,} rows in step 3! ***")

# Show position distribution
print("\n   Position distribution:")
player_events.groupBy('position_bucket').count().orderBy('count', ascending=False).show()

In [ ]:
# STEP 4: Opponent context features
print("\n[4/8] Computing opponent context features...")

# ============================================================================
# VERIFICATION: Track row count at start of step
# ============================================================================
rows_at_step_start = player_events.count()
print(f"   Rows at START of step 4: {rows_at_step_start:,}")

# ============================================================================
# FIX: Use action_no_in_hand from SP-2 as idx (correct action sequence)
# DO NOT use monotonically_increasing_id() - it does NOT preserve order!
# ============================================================================
if 'idx' not in player_events.columns:
    if 'action_no_in_hand' in player_events.columns:
        print("   Using action_no_in_hand from SP-2 as idx (correct sequence)")
        player_events = player_events.withColumn('idx', F.col('action_no_in_hand'))
    else:
        print("   WARNING: action_no_in_hand not found - creating idx from street order")
        # Fallback: order by street_rank then within-street sequence
        street_order = {'preflop': 0, 'flop': 1, 'turn': 2, 'river': 3, 'showdown': 4}
        street_order_expr = F.create_map([F.lit(x) for item in street_order.items() for x in item])
        
        player_events = player_events.withColumn('_street_rank', street_order_expr[F.col('street')])
        action_seq_window = Window.partitionBy('hand_id').orderBy('_street_rank', F.monotonically_increasing_id())
        player_events = player_events.withColumn('idx', F.row_number().over(action_seq_window))
        player_events = player_events.drop('_street_rank')

# Verify idx ordering
print(f"\n[DEBUG] Verifying idx ordering for sample hand:")
sample_hand = player_events.select('hand_id').first()['hand_id']
player_events.filter(F.col('hand_id') == sample_hand).select(
    'hand_id', 'idx', 'street', 'actor', 'action_type'
).orderBy('idx').show(15, truncate=False)

# Track active players using cumulative fold count
action_window = Window.partitionBy('hand_id').orderBy('idx')

player_events = player_events.withColumn(
    'is_fold_action',
    F.when(F.col('action_type') == 'fold', 1).otherwise(0)
)

player_events = player_events.withColumn(
    'cumulative_folds',
    F.sum('is_fold_action').over(action_window)
)

player_events = player_events.withColumn(
    'folds_before',
    F.lag('cumulative_folds', 1, 0).over(action_window)
)

# Get total players per hand (if not already present)
if 'players_at_table' not in player_events.columns:
    total_players = player_events.groupBy('hand_id').agg(
        F.countDistinct('actor').alias('players_at_table')
    )
    rows_before = player_events.count()
    player_events = player_events.join(total_players, on='hand_id', how='left')
    rows_after = player_events.count()
    print(f"   After total_players join: {rows_before:,} -> {rows_after:,}")

player_events = player_events.withColumn(
    'players_active',
    F.col('players_at_table') - F.col('folds_before')
)

player_events = player_events.withColumn(
    'opponents_active',
    F.greatest(F.col('players_active') - 1, F.lit(0))
)

# Stack statistics per hand
if 'starting_stack' in player_events.columns:
    stack_stats = player_events.groupBy('hand_id').agg(
        F.avg('starting_stack').alias('_avg_stack'),
        F.max('starting_stack').alias('_max_stack'),
        F.min('starting_stack').alias('_min_stack'),
        F.sum('starting_stack').alias('_total_stack')
    )

    rows_before = player_events.count()
    player_events = player_events.join(stack_stats, on='hand_id', how='left')
    rows_after = player_events.count()
    print(f"   After stack_stats join: {rows_before:,} -> {rows_after:,}")

    player_events = player_events.withColumn(
        'avg_opponent_stack',
        F.when(F.col('opponents_active') > 0,
               (F.col('_total_stack') - F.col('starting_stack')) / F.col('opponents_active'))
        .otherwise(0)
    )

    player_events = player_events.withColumn('max_opponent_stack', F.col('_max_stack'))
    player_events = player_events.withColumn('min_opponent_stack', F.col('_min_stack'))

    # Clean up temp columns
    player_events = player_events.drop('_avg_stack', '_max_stack', '_min_stack', '_total_stack')
else:
    print("   starting_stack not found - using defaults")
    player_events = player_events.withColumn('avg_opponent_stack', F.lit(100.0))
    player_events = player_events.withColumn('max_opponent_stack', F.lit(100.0))
    player_events = player_events.withColumn('min_opponent_stack', F.lit(100.0))

# Position counts by hand - use unique prefix to avoid column conflicts
pos_counts = player_events.groupBy('hand_id').pivot('position_bucket').count()

# Rename pivot columns with unique prefix
renamed_cols = ['hand_id']
for col_name in pos_counts.columns:
    if col_name != 'hand_id':
        renamed_cols.append(f'pos_count_{col_name}')

pos_counts = pos_counts.toDF(*renamed_cols)

# Join position counts
rows_before = player_events.count()
player_events = player_events.join(pos_counts, on='hand_id', how='left')
rows_after = player_events.count()
print(f"   After pos_counts join: {rows_before:,} -> {rows_after:,}")

# Fill nulls for position counts
position_cols = [c for c in player_events.columns if c.startswith('pos_count_')]
for col in position_cols:
    player_events = player_events.fillna({col: 0})

# Clean up temporary columns
player_events = player_events.drop('is_fold_action', 'cumulative_folds', 'folds_before')

print("   Added opponent context features")
rows_at_step_end = player_events.count()
print(f"   Rows at END of step 4: {rows_at_step_end:,}")
if rows_at_step_end != rows_at_step_start:
    print(f"   *** WARNING: Lost {rows_at_step_start - rows_at_step_end:,} rows in step 4! ***")

In [ ]:
# STEP 4: (Reserved for future use - cells were consolidated)
# This cell intentionally left minimal to avoid duplicate processing
print("\n[4/8] Step 4 reserved - opponent context computed in previous cell")

In [ ]:
# STEP 5: Board texture features
print("\n[5/8] Computing board texture features...")

# ============================================================================
# VERIFICATION: Track row count at start of step
# ============================================================================
rows_at_step_start = player_events.count()
print(f"   Rows at START of step 5: {rows_at_step_start:,}")

# Check if we have raw board columns OR the derived board features from SP-2
board_cols = ['flop', 'turn', 'river']
derived_board_cols = ['board_flush_possible', 'board_pair_or_better', 'board_straight_possible']

has_raw_board = all(c in player_events.columns for c in board_cols)
has_derived_board = all(c in player_events.columns for c in derived_board_cols)

if has_raw_board:
    print("   Using raw board cards to compute texture features...")
    
    # Create UDF for board texture analysis
    @F.udf(returnType=StructType([
        StructField('board_monotone', IntegerType()),
        StructField('board_paired', IntegerType()),
        StructField('board_straighty', IntegerType())
    ]))
    def analyze_board_udf(street, flop, turn, river):
        # Build visible board based on street
        board_str = ''
        if street in ('flop', 'turn', 'river') and flop:
            board_str = flop
        if street in ('turn', 'river') and turn:
            board_str += turn
        if street == 'river' and river:
            board_str += river
        
        if not board_str or len(board_str) < 6:
            return (0, 0, 0)
        
        # Parse cards
        cards = [board_str[i:i+2] for i in range(0, len(board_str), 2) if len(board_str[i:i+2]) == 2]
        if len(cards) < 3:
            return (0, 0, 0)
        
        suits = [c[1].lower() for c in cards]
        ranks = [c[0].upper() for c in cards]
        
        # Monotone (all same suit)
        mono = 1 if len(set(suits)) == 1 else 0
        
        # Paired
        paired = 1 if any(ranks.count(r) >= 2 for r in set(ranks)) else 0
        
        # Straighty (connected)
        rank_val = {'2':2,'3':3,'4':4,'5':5,'6':6,'7':7,'8':8,'9':9,'T':10,'J':11,'Q':12,'K':13,'A':14}
        vals = {rank_val.get(r, 0) for r in ranks}
        if 14 in vals:
            vals.add(1)  # Ace can be low
        straighty = 1 if any(len(vals & set(range(s, s+3))) == 3 for s in range(1, 12)) else 0
        
        return (mono, paired, straighty)

    # Apply board analysis
    board_features = analyze_board_udf(
        F.col('street'),
        F.col('flop'),
        F.col('turn'),
        F.col('river')
    )

    player_events = player_events.withColumn('_board_features', board_features)
    player_events = player_events.withColumn('board_monotone', F.col('_board_features.board_monotone'))
    player_events = player_events.withColumn('board_paired', F.col('_board_features.board_paired'))
    player_events = player_events.withColumn('board_straighty', F.col('_board_features.board_straighty'))
    player_events = player_events.drop('_board_features')

    # Additional texture features
    player_events = player_events.withColumn('board_flush_pressure', F.col('board_monotone'))
    player_events = player_events.withColumn('board_straight_pressure', F.col('board_straighty'))
    
    print("   Added board texture features from raw cards")

elif has_derived_board:
    print("   Using pre-computed board features from SP-2...")
    print(f"   Available: {derived_board_cols}")
    
    # Map SP-2 features to our expected column names
    player_events = player_events.withColumn(
        'board_monotone', 
        F.col('board_flush_possible').cast(IntegerType())
    )
    player_events = player_events.withColumn(
        'board_paired', 
        F.col('board_pair_or_better').cast(IntegerType())
    )
    player_events = player_events.withColumn(
        'board_straighty', 
        F.col('board_straight_possible').cast(IntegerType())
    )
    
    # Additional texture features
    player_events = player_events.withColumn('board_flush_pressure', F.col('board_flush_possible').cast(IntegerType()))
    player_events = player_events.withColumn('board_straight_pressure', F.col('board_straight_possible').cast(IntegerType()))
    
    print("   Mapped SP-2 board features to standard column names")

else:
    print("\n" + "=" * 80)
    print("WARNING: No board texture data available!")
    print("=" * 80)
    print(f"\nMissing raw board columns: {[c for c in board_cols if c not in player_events.columns]}")
    print(f"Missing derived columns: {[c for c in derived_board_cols if c not in player_events.columns]}")
    print("\nUsing default values (0) for board texture features.")
    
    # Add default columns
    player_events = player_events.withColumn('board_monotone', F.lit(0))
    player_events = player_events.withColumn('board_paired', F.lit(0))
    player_events = player_events.withColumn('board_straighty', F.lit(0))
    player_events = player_events.withColumn('board_flush_pressure', F.lit(0))
    player_events = player_events.withColumn('board_straight_pressure', F.lit(0))

print("   Board texture features ready")

# ============================================================================
# VERIFICATION: Check row count at end of step
# ============================================================================
rows_at_step_end = player_events.count()
print(f"   Rows at END of step 5: {rows_at_step_end:,}")
if rows_at_step_end != rows_at_step_start:
    print(f"   *** WARNING: Lost {rows_at_step_start - rows_at_step_end:,} rows in step 5! ***")

In [ ]:
# STEP 6: Join pre-computed opponent predictions from SP-3
print("\n[6/8] Joining SP-3 opponent predictions...")

# ============================================================================
# VERIFICATION: Track row count before and after joins
# ============================================================================
rows_before_prediction_join = player_events.count()
print(f"   Rows BEFORE prediction join: {rows_before_prediction_join:,}")

if SP3_PREDICTIONS_LOADED and sp3_predictions is not None:
    print(f"   Using pre-computed predictions from SP-3")
    
    # ========================================================================
    # FIX: Use action_no_in_hand as idx if idx doesn't exist
    # DO NOT use monotonically_increasing_id() - it does NOT preserve order!
    # ========================================================================
    if 'idx' not in player_events.columns:
        if 'action_no_in_hand' in player_events.columns:
            print("   Using action_no_in_hand from SP-2 as idx")
            player_events = player_events.withColumn('idx', F.col('action_no_in_hand'))
        else:
            print("   WARNING: Creating idx from street order (fallback)")
            street_order = {'preflop': 0, 'flop': 1, 'turn': 2, 'river': 3, 'showdown': 4}
            street_order_expr = F.create_map([F.lit(x) for item in street_order.items() for x in item])
            player_events = player_events.withColumn('_street_rank', street_order_expr[F.col('street')])
            action_seq_window = Window.partitionBy('hand_id').orderBy('_street_rank', F.monotonically_increasing_id())
            player_events = player_events.withColumn('idx', F.row_number().over(action_seq_window))
            player_events = player_events.drop('_street_rank')
    
    # Join predictions to player_events
    # SP-3 predictions have: hand_id, idx, actor, street, predicted_bucket, predicted_strength
    player_events = player_events.join(
        sp3_predictions.select('hand_id', 'idx', 'predicted_bucket', 'predicted_strength'),
        on=['hand_id', 'idx'],
        how='left'
    )
    
    # ========================================================================
    # VERIFICATION: Check row count after prediction join
    # ========================================================================
    rows_after_prediction_join = player_events.count()
    print(f"   Rows AFTER prediction join: {rows_after_prediction_join:,}")
    
    if rows_after_prediction_join != rows_before_prediction_join:
        print(f"   *** WARNING: Row count changed! Lost {rows_before_prediction_join - rows_after_prediction_join:,} rows ***")
        print(f"   This may indicate duplicate keys in predictions causing row explosion or loss")
    else:
        print(f"   ✓ Row count preserved after prediction join")
    
    # Fill missing predictions with defaults
    player_events = player_events.fillna({
        'predicted_strength': 0.5,
        'predicted_bucket': 'middle'
    })
    
    # Check join success
    non_null_preds = player_events.filter(F.col('predicted_bucket').isNotNull()).count()
    total_rows = player_events.count()
    print(f"   Predictions joined: {non_null_preds:,}/{total_rows:,} ({100*non_null_preds/total_rows:.1f}%)")
    
    # Compute opponent aggregates
    print("\n   Computing opponent prediction aggregates...")
    
    rows_before_opp_join = player_events.count()
    
    # Get opponent predictions up to current action
    player_events_alias = player_events.select(
        F.col('hand_id').alias('h_id'),
        F.col('idx').alias('action_idx'),
        F.col('actor').alias('current_actor')
    )
    
    # Join predictions for opponent analysis
    opponent_preds = sp3_predictions.alias('opp').join(
        player_events_alias.alias('main'),
        (F.col('opp.hand_id') == F.col('main.h_id')) &
        (F.col('opp.idx') <= F.col('main.action_idx')) &
        (F.col('opp.actor') != F.col('main.current_actor')),
        how='inner'
    ).select(
        F.col('main.h_id').alias('hand_id'),
        F.col('main.action_idx').alias('idx'),
        F.col('opp.predicted_strength').alias('opp_strength'),
        F.col('opp.predicted_bucket').alias('opp_bucket')
    )
    
    # Aggregate opponent predictions
    opp_agg = opponent_preds.groupBy('hand_id', 'idx').agg(
        F.avg('opp_strength').alias('opponent_strength_mean'),
        F.max('opp_strength').alias('opponent_strength_max'),
        F.min('opp_strength').alias('opponent_strength_min'),
        F.sum(F.when(F.col('opp_bucket') == 'air', 1).otherwise(0)).alias('opponent_air_count'),
        F.sum(F.when(F.col('opp_bucket') == 'middle', 1).otherwise(0)).alias('opponent_middle_count'),
        F.sum(F.when(F.col('opp_bucket') == 'nutted', 1).otherwise(0)).alias('opponent_nutted_count'),
        F.lit(1).alias('has_opponent_predictions')
    )
    
    player_events = player_events.join(opp_agg, on=['hand_id', 'idx'], how='left')
    
    # ========================================================================
    # VERIFICATION: Check row count after opponent aggregate join
    # ========================================================================
    rows_after_opp_join = player_events.count()
    print(f"   Rows AFTER opponent aggregate join: {rows_after_opp_join:,}")
    
    if rows_after_opp_join != rows_before_opp_join:
        print(f"   *** WARNING: Row count changed! Lost {rows_before_opp_join - rows_after_opp_join:,} rows ***")
    else:
        print(f"   ✓ Row count preserved after opponent aggregate join")
    
    # Fill missing aggregates
    player_events = player_events.fillna({
        'opponent_strength_mean': 0.5,
        'opponent_strength_max': 0.5,
        'opponent_strength_min': 0.5,
        'opponent_air_count': 0,
        'opponent_middle_count': 0,
        'opponent_nutted_count': 0,
        'has_opponent_predictions': 0
    })
    
    print("   Added opponent prediction features")
    
else:
    print("   WARNING: SP-3 predictions not available - using defaults")
    print("   Run notebook 03 first to generate predictions")
    
    player_events = player_events.withColumn('predicted_strength', F.lit(0.5))
    player_events = player_events.withColumn('predicted_bucket', F.lit('middle'))
    player_events = player_events.withColumn('opponent_strength_mean', F.lit(0.5))
    player_events = player_events.withColumn('opponent_strength_max', F.lit(0.5))
    player_events = player_events.withColumn('opponent_strength_min', F.lit(0.5))
    player_events = player_events.withColumn('opponent_air_count', F.lit(0))
    player_events = player_events.withColumn('opponent_middle_count', F.lit(0))
    player_events = player_events.withColumn('opponent_nutted_count', F.lit(0))
    player_events = player_events.withColumn('has_opponent_predictions', F.lit(0))

# ============================================================================
# FINAL VERIFICATION
# ============================================================================
final_row_count = player_events.count()
print(f"\n   FINAL row count: {final_row_count:,}")
print(f"   Net change from start of step: {final_row_count - rows_before_prediction_join:,}")

if final_row_count != rows_before_prediction_join:
    print(f"\n   *** DATA INTEGRITY WARNING ***")
    print(f"   Started with {rows_before_prediction_join:,} rows, ended with {final_row_count:,}")
else:
    print(f"   ✓ Data integrity preserved - no row loss")

# ============================================================================
# TRACE: After opponent predictions
# ============================================================================
print(f"\n[TRACE] After opponent predictions for {TRACE_HAND_ID}:")
player_events.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'idx', 'actor', 'street', 'action_type',
    'predicted_bucket', 'predicted_strength',
    'opponent_strength_mean', 'has_opponent_predictions'
).orderBy('idx').show(20, truncate=False)

In [ ]:
# STEP 8: Pot odds and betting features
print("\n[8/8] Computing pot odds and betting features...")

# Ensure amount column exists and is numeric
if 'amount' in player_events.columns:
    player_events = player_events.withColumn(
        'amount',
        F.coalesce(F.col('amount').cast(DoubleType()), F.lit(0.0))
    )
else:
    player_events = player_events.withColumn('amount', F.lit(0.0))

# Calculate pot size progression
pot_window = Window.partitionBy('hand_id').orderBy('idx')

player_events = player_events.withColumn(
    'pot_contribution',
    F.col('amount')
)

player_events = player_events.withColumn(
    'pot_size',
    F.sum('pot_contribution').over(pot_window)
)

player_events = player_events.withColumn(
    'pot_before_action',
    F.lag('pot_size', 1, 0).over(pot_window)
)

# Calculate street contributions for pot odds
street_window = Window.partitionBy('hand_id', 'street').orderBy('idx')

player_events = player_events.withColumn(
    'street_max_bet',
    F.max('amount').over(street_window)
)

player_events = player_events.withColumn(
    'street_max_bet_before',
    F.lag('street_max_bet', 1, 0).over(street_window)
)

# Facing call amount (simplified)
player_events = player_events.withColumn(
    'facing_call',
    F.greatest(F.col('street_max_bet_before') - F.col('amount'), F.lit(0.0))
)

# Pot odds
player_events = player_events.withColumn(
    'pot_odds_call',
    F.when(
        (F.col('facing_call') > 0) & (F.col('pot_before_action') + F.col('facing_call') > 0),
        F.col('facing_call') / (F.col('pot_before_action') + F.col('facing_call'))
    ).otherwise(0.0)
)

# Clean up temporary columns
player_events = player_events.drop('street_max_bet', 'street_max_bet_before', 'pot_contribution')

print("   Added pot odds and betting features")

In [ ]:
# ============================================================================
# STEP 8B: COMPUTE PROFIT TARGETS (Required for SP-5 Profit Model)
# ============================================================================
print("\n[8B/9] Computing profit targets from hand outcomes...")

rows_at_start = player_events.count()
print(f"   Rows at START: {rows_at_start:,}")

# Profit calculation: target_profit_bb = (payout - contribution) / bb
# This was in the old SP-4-FeatureEngineeringFaster.py but missing from PySpark version

# ============================================================================
# Load hands table to get BB size
# ============================================================================
CHUNKS_DIR = '/Workspace/Users/leo.lwakabamba@gmail.com/poker-ml-data/chunks/'

try:
    hands_df = spark.read.parquet(f"{CHUNKS_DIR}phh_multi_hands_chunk_*.parquet")
    print(f"   Loaded hands table: {hands_df.count():,} rows")
    
    if 'bb' in hands_df.columns:
        # Deduplicate hands_df by hand_id (take first bb value)
        # Also ensure bb is never 0 to avoid division by zero
        bb_lookup = hands_df.select(
            'hand_id', 
            F.when(F.col('bb').cast('float') > 0, F.col('bb').cast('float')).otherwise(1.0).alias('bb')
        ).dropDuplicates(['hand_id'])
        
        rows_before = player_events.count()
        player_events = player_events.join(bb_lookup, on='hand_id', how='left')
        rows_after = player_events.count()
        print(f"   After BB join: {rows_before:,} -> {rows_after:,}")
        
        player_events = player_events.fillna({'bb': 1.0})
        print("   BB size joined from hands table")
    else:
        print("   WARNING: 'bb' column not in hands table - defaulting to 1.0")
        player_events = player_events.withColumn('bb', F.lit(1.0))
except Exception as e:
    print(f"   WARNING: Could not load hands table: {e}")
    print("   Using default BB = 1.0")
    player_events = player_events.withColumn('bb', F.lit(1.0))

# ============================================================================
# Compute total contribution per player per hand
# ============================================================================
contributions = player_events.groupBy('hand_id', 'actor').agg(
    F.sum('amount').alias('total_contribution')
)

# Get final pot per hand (max pot_size in hand)
final_pots = player_events.groupBy('hand_id').agg(
    F.max('pot_size').alias('final_pot')
)

# ============================================================================
# Determine winners using hand_equity at river/showdown
# Higher hand_equity = better hand = winner
# ============================================================================
print("   Determining winners based on hand_equity...")

# Get each player's best hand_equity in the hand (at river is most accurate)
player_strength = player_events.filter(
    F.col('hand_equity').isNotNull()
).groupBy('hand_id', 'actor').agg(
    F.max('hand_equity').alias('final_strength')
)

# Find the winning strength per hand
winning_strength = player_strength.groupBy('hand_id').agg(
    F.max('final_strength').alias('best_strength')
)

# Mark winners (players whose final_strength equals best_strength)
player_with_winner = player_strength.join(winning_strength, on='hand_id', how='left')
player_with_winner = player_with_winner.withColumn(
    'is_winner',
    F.when(
        (F.col('final_strength') == F.col('best_strength')) & 
        (F.col('final_strength').isNotNull()),
        1
    ).otherwise(0)
)

# Count winners per hand (for split pots)
winner_counts = player_with_winner.filter(F.col('is_winner') == 1).groupBy('hand_id').agg(
    F.count('*').alias('num_winners')
)

# Join all together to compute payouts
profit_calc = player_with_winner.join(winner_counts, on='hand_id', how='left')
profit_calc = profit_calc.join(final_pots, on='hand_id', how='left')
profit_calc = profit_calc.join(contributions, on=['hand_id', 'actor'], how='left')

# Fill nulls - ensure num_winners is never 0 to avoid division by zero
profit_calc = profit_calc.fillna({
    'num_winners': 1,
    'final_pot': 0.0,
    'total_contribution': 0.0
})

# Ensure num_winners is at least 1 (safety check for division)
profit_calc = profit_calc.withColumn(
    'num_winners',
    F.when(F.col('num_winners') < 1, 1).otherwise(F.col('num_winners'))
)

# Calculate payout and profit - use safe division
profit_calc = profit_calc.withColumn(
    'payout',
    F.when(
        (F.col('is_winner') == 1) & (F.col('num_winners') > 0),
        F.col('final_pot') / F.col('num_winners')
    ).otherwise(0.0)
)
profit_calc = profit_calc.withColumn(
    'profit_chips',
    F.col('payout') - F.col('total_contribution')
)

# Select just what we need for joining back - DEDUPLICATE to ensure one row per (hand_id, actor)
profit_lookup = profit_calc.select('hand_id', 'actor', 'profit_chips', 'is_winner').dropDuplicates(['hand_id', 'actor'])

# Verify no duplicates
lookup_count = profit_lookup.count()
unique_count = profit_lookup.select('hand_id', 'actor').distinct().count()
print(f"   Profit lookup: {lookup_count:,} rows, {unique_count:,} unique (hand_id, actor) pairs")
if lookup_count != unique_count:
    print(f"   *** WARNING: Duplicates in profit_lookup! ***")

# Join profit back to player_events
rows_before = player_events.count()
player_events = player_events.join(profit_lookup, on=['hand_id', 'actor'], how='left')
rows_after = player_events.count()
print(f"   After profit join: {rows_before:,} -> {rows_after:,}")

if rows_after != rows_before:
    print(f"   *** WARNING: Row count changed! Investigating... ***")
    # This shouldn't happen with left join and deduplicated lookup

# Compute target_profit_bb - use safe division (bb should never be 0, but guard anyway)
player_events = player_events.withColumn(
    'target_profit_bb',
    F.when(
        F.col('bb') > 0,
        F.col('profit_chips') / F.col('bb')
    ).otherwise(0.0)
)

# Fill nulls
player_events = player_events.fillna({
    'profit_chips': 0.0,
    'target_profit_bb': 0.0,
    'is_winner': 0
})

# ============================================================================
# TRACE: Profit calculation for TRACE_HAND_ID (with idx for ordering verification)
# ============================================================================
print(f"\n[TRACE] Profit calculation for {TRACE_HAND_ID} (ordered by idx):")
player_events.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'idx', 'actor', 'street', 'action_type', 'hand_equity', 'is_winner', 'profit_chips', 'target_profit_bb'
).orderBy('idx').show(20, truncate=False)

# Summary stats
winners_count = player_events.filter(F.col('is_winner') == 1).select('hand_id', 'actor').distinct().count()
print(f"   Total unique winners: {winners_count:,}")

# Profit distribution - filter out nulls for summary
print("\n   Profit distribution (BB):")
profit_stats = player_events.filter(
    F.col('target_profit_bb').isNotNull()
).select('target_profit_bb').summary('mean', 'min', 'max')
profit_stats.show()

rows_at_end = player_events.count()
print(f"   Rows at END: {rows_at_end:,}")
if rows_at_end != rows_at_start:
    print(f"   *** WARNING: Net row change: {rows_at_end - rows_at_start:,} ***")
else:
    print(f"   ✓ Row count preserved")

print("   Profit targets computed successfully")

In [ ]:
# ============================================================================
# STEP 8C: NORMALIZE ALL CHIP VALUES TO BB
# ============================================================================
# CRITICAL FIX: All chip-based features must be in BB units for model consistency
# The model predicts target_profit_bb (in BB), so features should also be in BB
# 
# Without this fix, the model sees:
#   - pot_size = 500 (chips) when it should see 5 (BB with bb=100)
#   - starting_stack = 10000 (chips) when it should see 100 (BB)
# This causes predictions like -504 BB instead of -5 BB
# ============================================================================

print("\n[8C/9] Normalizing chip values to BB units...")

rows_before = player_events.count()

# List of features that need BB normalization (currently in chips)
CHIP_FEATURES_TO_NORMALIZE = [
    'pot_size',
    'starting_stack', 
    'amount',
    'avg_opponent_stack',
    'max_opponent_stack', 
    'min_opponent_stack',
    'pot_before_action',
    'facing_call',
]

# Normalize each feature
for feature in CHIP_FEATURES_TO_NORMALIZE:
    if feature in player_events.columns:
        # Create BB-normalized version (divide chips by bb)
        player_events = player_events.withColumn(
            feature,
            F.when(
                F.col('bb') > 0,
                F.col(feature) / F.col('bb')
            ).otherwise(F.col(feature))  # Keep original if bb is 0/null (shouldn't happen)
        )
        print(f"   ✓ Normalized {feature} to BB")
    else:
        print(f"   - Skipped {feature} (not in dataset)")

# Verify normalization worked
print("\n   Verification - checking value ranges after normalization:")

# Sample stats for key features
for feature in ['pot_size', 'starting_stack', 'avg_opponent_stack']:
    if feature in player_events.columns:
        stats = player_events.filter(F.col(feature).isNotNull()).select(
            F.min(feature).alias('min'),
            F.max(feature).alias('max'),
            F.avg(feature).alias('mean')
        ).collect()[0]
        print(f"   {feature:25s}: min={stats['min']:.1f}, max={stats['max']:.1f}, mean={stats['mean']:.1f} BB")

# Also normalize stack_trend columns if they exist
stack_trend_cols = [c for c in player_events.columns if 'stack_trend' in c]
if stack_trend_cols:
    print(f"\n   Normalizing stack_trend columns: {stack_trend_cols}")
    for col in stack_trend_cols:
        player_events = player_events.withColumn(
            col,
            F.when(
                F.col('bb') > 0,
                F.col(col) / F.col('bb')
            ).otherwise(F.col(col))
        )
        print(f"   ✓ Normalized {col} to BB")

rows_after = player_events.count()
print(f"\n   Rows: {rows_before:,} -> {rows_after:,}")

if rows_after != rows_before:
    print(f"   *** WARNING: Row count changed! ***")
else:
    print(f"   ✓ Row count preserved")

print("\n   BB normalization complete - all chip features now in BB units")

In [ ]:
# Save output
print("\nSaving SP-4 features...")

# Drop columns we don't need in output
drop_cols = ['hand_order', 'voluntary_preflop', 'preflop_raise', 'position_pct']
output_df = player_events.drop(*[c for c in drop_cols if c in player_events.columns])

# ============================================================================
# VERIFY PROFIT COLUMNS EXIST BEFORE SAVE
# ============================================================================
print("\n   Verifying profit columns before save:")
profit_cols = ['target_profit_bb', 'profit_chips', 'is_winner', 'bb']
for col in profit_cols:
    exists = col in output_df.columns
    print(f"      {col}: {'✓' if exists else '*** MISSING ***'}")

if 'target_profit_bb' not in output_df.columns:
    print("\n   *** ERROR: target_profit_bb missing! Check step 8B ***")

# ============================================================================
# TRACE: FINAL OUTPUT for TRACE_HAND_ID
# ============================================================================
print(f"\n[TRACE] FINAL OUTPUT for {TRACE_HAND_ID}:")
output_df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'action_type',
    'target_profit_bb', 'is_winner',
    'position_from_button', 'position_name',
    'predicted_bucket', 'pot_odds_call'
).show(20, truncate=False)

# Save as Parquet
output_df.write.mode('overwrite').parquet(OUTPUT_PATH)

# Get final stats
final_count = output_df.count()
final_cols = len(output_df.columns)

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("SP-4 Feature Engineering COMPLETE!")
print("=" * 80)
print(f"\nRuntime: {elapsed/60:.1f} minutes")
print(f"Output: {OUTPUT_PATH}")
print(f"Dataset: {final_count:,} rows, {final_cols} columns")
print(f"\n[TRACE] TRACE_HAND_ID used: {TRACE_HAND_ID}")
print(f"\nData Lineage:")
print(f"  Input: SP-2 output ({SP2_INPUT_PATH})")
print(f"  Predictions: SP-3 pre-computed ({SP3_PREDICTIONS_PATH})")
print(f"  Predictions loaded: {SP3_PREDICTIONS_LOADED}")
print(f"  Output: {OUTPUT_PATH}")
print(f"\nProfit columns saved: {[c for c in profit_cols if c in output_df.columns]}")
print(f"\nNext step: Run SP-5 (05_ProfitModel) to train profit prediction models")

In [ ]:
# Show sample of output including profit columns
print("\nSample output:")
output_df.select(
    'hand_id', 'actor', 'street', 'action_type',
    'target_profit_bb', 'is_winner', 'profit_chips',
    'predicted_strength', 'predicted_bucket',
    'pot_odds_call'
).show(10, truncate=False)

# Verify profit columns exist
print("\nProfit column verification:")
print(f"   target_profit_bb in output: {'target_profit_bb' in output_df.columns}")
print(f"   profit_chips in output: {'profit_chips' in output_df.columns}")
print(f"   is_winner in output: {'is_winner' in output_df.columns}")

---
## OLD PANDAS/SKLEARN VERSION (Commented Out)
The code below is the original pandas-based implementation that used sklearn models.
Kept for reference.

In [ ]:
# # OLD VERSION - PANDAS/SKLEARN IMPLEMENTATION
# # ============================================
# 
# # SP-4: Advanced Feature Engineering Pipeline
# # Combines full feature engineering with opponent modeling and hand strength
# 
# import pandas as pd
# import numpy as np
# import joblib
# import json
# import time
# from pathlib import Path
# from collections import defaultdict
# from treys import Card, Evaluator
# import warnings
# warnings.filterwarnings('ignore')
# 
# # ... (rest of original pandas implementation)
# # See git history for full original code